# Zero-copy clone and table metadata on AIDP Delta

Two Delta capabilities you can rely on in AIDP, demonstrated with explicit expected counts:

1. **Zero-copy clone** — `SHALLOW CLONE` creates a table that references the source's data files
   instead of copying them, and diverges from the source copy-on-write as soon as you write to it.
2. **Table and column metadata in SQL** — attaching and reading back key/value metadata via
   `TBLPROPERTIES`, column comments, and a registry table that scales to column scope.

## Prerequisites

- An AIDP cluster with Delta (tested on Spark 3.5.0 / Delta 3.2.0-oci-1.0.0).
- A catalog you can create schemas in. Set `CATALOG` below; the notebook creates a scratch schema
  and drops it at the end.

Every statement runs directly — there is no error-swallowing wrapper, so anything unsupported on your
build fails loudly at that cell rather than being recorded as a result.

In [ ]:
# Configuration -- point CATALOG at a catalog you can write to.
CATALOG = "default"
DB = "clone_metadata_demo"      # scratch schema; created here, dropped in the last cell

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{DB}")

# Set the current catalog and schema. Some Spark 3.5 grammar rules -- notably
# CREATE TABLE ... LIKE -- accept at most a two-part name, so later cells rely on
# these being current rather than fully qualifying every reference.
spark.sql(f"USE {CATALOG}.{DB}")
print(f"Working in {CATALOG}.{DB}")

In [ ]:
# A small table to clone and tag.
spark.sql("DROP TABLE IF EXISTS customers")
spark.sql("""
    CREATE TABLE customers (id INT, name STRING, email STRING, ssn STRING)
    USING delta
""")
spark.sql("""
    INSERT INTO customers VALUES
        (1, 'Alice', 'alice@example.com', '111-22-3333'),
        (2, 'Bob',   'bob@example.com',   '444-55-6666')
""")
spark.table("customers").show()

## 1. Zero-copy clone

`SHALLOW CLONE` copies only metadata: the new table's transaction log points at the **source's**
existing parquet files, so the clone is created in constant time regardless of table size. Writes to
either table create new files belonging to that table, leaving the other unchanged -- copy-on-write
divergence.

> **Operational warning.** A shallow clone depends on the source's data files. Running `VACUUM` on the
> source can delete files the clone still references and break it. Delta does not track that
> dependency for you, so either avoid vacuuming a cloned source or raise its retention window.

In [ ]:
# Create the clone. Constant time: no data is copied.
spark.sql("CREATE OR REPLACE TABLE customers_shallow SHALLOW CLONE customers")

print("rows in clone  (expect 2):", spark.table("customers_shallow").count())
print("rows in source (expect 2):", spark.table("customers").count())

In [ ]:
# DESCRIBE DETAIL confirms the zero-copy property. Note it is run as a top-level
# statement and projected with the DataFrame API: Spark 3.5 does not parse
# SELECT ... FROM (DESCRIBE DETAIL t), which is a syntax error rather than an
# unsupported feature.
(spark.sql("DESCRIBE DETAIL customers_shallow")
      .select("format", "numFiles", "sizeInBytes", "location")
      .show(truncate=False))

# The clone's first commit records the CLONE operation.
(spark.sql("DESCRIBE HISTORY customers_shallow")
      .select("version", "operation")
      .orderBy("version")
      .show(3, truncate=False))

In [ ]:
# Copy-on-write divergence: write to the clone, source is untouched.
spark.sql("INSERT INTO customers_shallow VALUES (99, 'Zoe', 'zoe@example.com', '999-88-7777')")

print("clone after insert  (expect 3):", spark.table("customers_shallow").count())
print("source unchanged    (expect 2):", spark.table("customers").count())

### Deep clone

`DEEP CLONE` copies the data files as well, producing a fully independent table -- useful, but not
zero-copy, and it costs time proportional to table size.

It is not exercised here because availability differs by build: the open-source Delta 3.2 grammar
rejects `DEEP CLONE` (the OSS docs document shallow only), while Oracle's `3.2.0-oci` build may
accept it. Running it unconditionally would halt this notebook partway through on a build that does
not, so if you want it, add `CREATE OR REPLACE TABLE t DEEP CLONE customers` and confirm against
your own build.

## 2. Cloning structure without data

Two ways to get an empty table with the same columns. `CREATE TABLE ... LIKE` copies the schema;
`CREATE TABLE ... AS SELECT ... WHERE 1=0` copies the schema of a query result.

`LIKE` accepts at most a two-part name in the Spark 3.5 grammar, which is why the configuration cell
set the current catalog and schema.

In [ ]:
spark.sql("DROP TABLE IF EXISTS customers_empty")
spark.sql("CREATE TABLE customers_empty LIKE customers")

print("LIKE clone rows    (expect 0):", spark.table("customers_empty").count())
print("LIKE clone columns (expect 4):", len(spark.table("customers_empty").columns))

spark.sql("DROP TABLE IF EXISTS customers_empty_ctas")
spark.sql("CREATE TABLE customers_empty_ctas AS SELECT * FROM customers WHERE 1=0")
print("CTAS-empty rows    (expect 0):", spark.table("customers_empty_ctas").count())

## 3. Table and column metadata in SQL

Delta and Spark SQL let you **attach** metadata at three granularities. None of it is *enforced*:
there is no policy engine that reads a tag and redacts a column at query time, so if you want
enforcement you build it yourself -- typically a view that redacts the columns your registry marks
sensitive.

| Mechanism | Scope | Notes |
|---|---|---|
| `TBLPROPERTIES` | table | A real key/value store on the table. |
| Column `COMMENT` | column | The only per-column SQL slot; free text, so one dimension at best. |
| Registry table | column | Metadata as data you query and join. Scales across the lakehouse. |

In [ ]:
# Table-scoped key/value metadata.
# Note 'owner' is a reserved table property in Spark, so use a distinct key.
spark.sql("""
    ALTER TABLE customers SET TBLPROPERTIES (
        'classification' = 'confidential',
        'data_owner'     = 'cdo',
        'pii'            = 'true'
    )
""")
spark.sql("SHOW TBLPROPERTIES customers").show(truncate=False)

In [ ]:
# Column-scoped metadata via the comment field. Spark has no
# ALTER COLUMN ... SET TAGS, so the comment is the only per-column slot.
spark.sql("ALTER TABLE customers ALTER COLUMN ssn COMMENT 'tag:pii;tag:masked'")

# DESCRIBE is run top-level and filtered with the DataFrame API, for the same
# parser reason as DESCRIBE DETAIL above.
spark.sql("DESCRIBE customers").show(truncate=False)

### A registry table for column metadata

Putting the metadata in a table makes it queryable: one governance table, ordinary `INSERT`s, and
questions like "which columns across the lakehouse are marked PII?" become plain SQL.

In [ ]:
# Dropped first so the INSERT below is not additive when the cell is re-run.
spark.sql("DROP TABLE IF EXISTS column_tags")
spark.sql("""
    CREATE TABLE IF NOT EXISTS column_tags (
        catalog_name STRING, schema_name STRING, table_name STRING,
        column_name  STRING, tag_key     STRING, tag_value  STRING
    ) USING delta
""")

spark.sql(f"""
    INSERT INTO column_tags VALUES
        ('{CATALOG}', '{DB}', 'customers', 'ssn',   'sensitivity', 'pii'),
        ('{CATALOG}', '{DB}', 'customers', 'email', 'sensitivity', 'pii')
""")

(spark.sql("""
    SELECT schema_name, table_name, column_name
    FROM column_tags
    WHERE tag_key = 'sensitivity' AND tag_value = 'pii'
    ORDER BY column_name
""").show(truncate=False))

## Cleanup

Drops the scratch schema and everything in it. The shallow clone and its source go together, so
there is no vacuum-ordering concern here.

In [ ]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{DB} CASCADE")
print(f"Dropped {CATALOG}.{DB}")